# Walmart Weekly Sales — Gradient Boosting Regressor

This notebook trains a **Gradient Boosting Regressor** to predict weekly sales across 45 Walmart stores using the [Walmart Dataset](https://www.kaggle.com/datasets/yasserh/walmart-dataset) from Kaggle.

**Pipeline overview:**
1. Load & inspect data
2. Feature engineering (date decomposition, encoding)
3. Statistical feature selection (carry over from linear regression notebook)
4. Train / test split + log-transform target
5. Train a Gradient Boosting model with GridSearchCV hyperparameter tuning
6. Evaluate with R², MAE, and RMSE
7. Interpret feature importances and diagnostic plots
8. Compare against Linear Regression baseline

## 1. Setup

In [ ]:
!pip install kagglehub -q

In [ ]:
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("Libraries loaded.")

## 2. Load Data

In [ ]:
import kagglehub

path = kagglehub.dataset_download('yasserh/walmart-dataset')
file_path = os.path.join(path, os.listdir(path)[0])

# --- Alternative: upload Walmart.csv manually and set ---
# file_path = 'Walmart.csv'

df = pd.read_csv(file_path)
print(f'Shape: {df.shape}')
df.head()

## 3. Feature Engineering

The `Date` column is decomposed into `Week`, `Month`, and `Year` to capture seasonality.
`Store` is one-hot encoded (drop_first=True to avoid the dummy variable trap).
`Fuel_Price` is dropped based on significance testing from the linear regression notebook (Pearson p = 0.4478).

In [ ]:
df['Date']  = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Week']  = df['Date'].dt.isocalendar().week.astype(int)
df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year
df.drop(columns=['Date'], inplace=True)

le = LabelEncoder()
df['Holiday_Flag'] = le.fit_transform(df['Holiday_Flag'])

print("Dtypes after feature engineering:")
print(df.dtypes)
df.describe()

## 4. Target Log-Transformation

`Weekly_Sales` is right-skewed with values ranging from ~$210k to ~$3.8M.
Log-transforming the target:
- Reduces the influence of extreme high-sales weeks
- Aligns with the Kaggle evaluation metric (RMSE on log-scale)
- Often improves regression model performance on skewed targets

We train on `log(Weekly_Sales)` and convert predictions back to the original scale for interpretability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Weekly_Sales'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Weekly_Sales (original scale)', fontsize=12)
axes[0].set_xlabel('Weekly Sales ($)')
axes[0].set_ylabel('Frequency')

log_sales = np.log(df['Weekly_Sales'])
axes[1].hist(log_sales, bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('log(Weekly_Sales)', fontsize=12)
axes[1].set_xlabel('log(Weekly Sales)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Target Variable Distribution: Before and After Log-Transform', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Original skewness: {df['Weekly_Sales'].skew():.4f}")
print(f"Log-transformed skewness: {log_sales.skew():.4f}")

## 5. Feature Selection & Encoding

In [ ]:
target = 'Weekly_Sales'

# Drop Fuel_Price (not significant, Pearson p=0.4478 from statistical analysis)
df.drop(columns=['Fuel_Price'], inplace=True)

# One-hot encode Store (nominal variable with 45 categories)
df_model = pd.get_dummies(df, columns=['Store'], drop_first=True)

X = df_model.drop(columns=[target])
y = np.log(df_model[target])   # log-transform target

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {X.columns.tolist()[:10]} ... (total {len(X.columns)})")

## 6. Train / Test Split

80/20 split with fixed random_state=42 for reproducibility and comparability with other notebooks.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  |  y_test: {y_test.shape}')

## 7. Gradient Boosting — Hyperparameter Tuning

Gradient Boosting builds an ensemble of decision trees sequentially, where each tree corrects
the residual errors of the previous one. Key hyperparameters:

| Parameter | Description |
|---|---|
| `n_estimators` | Number of boosting rounds (trees). More = better fit, but slower and risk of overfitting. |
| `max_depth` | Maximum depth of each individual tree. Controls model complexity. |
| `learning_rate` | Shrinkage factor applied to each tree's contribution. Lower = more conservative, needs more estimators. |
| `subsample` | Fraction of training samples used per tree. < 1.0 introduces stochastic gradient boosting, reducing variance. |

GridSearchCV with 5-fold cross-validation tunes these parameters. **Note:** Gradient Boosting does
not require feature scaling — tree-based models are scale-invariant.

In [ ]:
gb = GradientBoostingRegressor(random_state=42)

param_grid = {
    'n_estimators' : [100, 200, 300],
    'max_depth'    : [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample'    : [0.8, 1.0]
}

gb_grid = GridSearchCV(
    estimator  = gb,
    param_grid = param_grid,
    scoring    = 'r2',
    cv         = 5,
    n_jobs     = -1,
    verbose    = 1
)

gb_grid.fit(X_train, y_train)

print("\nBest parameters:")
for k, v in gb_grid.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV R²: {gb_grid.best_score_:.4f}")

## 8. Evaluate on Test Set

In [ ]:
best_gb = gb_grid.best_estimator_

y_pred_train_log = best_gb.predict(X_train)
y_pred_test_log  = best_gb.predict(X_test)

# Metrics on log scale (comparable to Kaggle benchmark)
train_r2_log  = r2_score(y_train, y_pred_train_log)
test_r2_log   = r2_score(y_test,  y_pred_test_log)
test_rmse_log = mean_squared_error(y_test, y_pred_test_log) ** 0.5

# Convert back to original scale for interpretability
y_pred_test_orig  = np.exp(y_pred_test_log)
y_test_orig       = np.exp(y_test)
y_pred_train_orig = np.exp(y_pred_train_log)
y_train_orig      = np.exp(y_train)

train_r2  = r2_score(y_train_orig, y_pred_train_orig)
test_r2   = r2_score(y_test_orig,  y_pred_test_orig)
test_mae  = mean_absolute_error(y_test_orig, y_pred_test_orig)
test_rmse = mean_squared_error(y_test_orig,  y_pred_test_orig) ** 0.5

print('=' * 50)
print('GRADIENT BOOSTING — Results (Log Scale)')
print('=' * 50)
print(f'  Train R²:         {train_r2_log:.4f}')
print(f'  Test  R²:         {test_r2_log:.4f}')
print(f'  Test  RMSE (log): {test_rmse_log:.4f}')
print()
print('=' * 50)
print('GRADIENT BOOSTING — Results (Original Scale)')
print('=' * 50)
print(f'  Train R²:   {train_r2:.4f}')
print(f'  Test  R²:   {test_r2:.4f}')
print(f'  Test  MAE:  ${test_mae:,.0f}')
print(f'  Test  RMSE: ${test_rmse:,.0f}')

## 9. Overfitting Check

In [ ]:
gap = train_r2_log - test_r2_log
print(f"Train R² (log): {train_r2_log:.4f}")
print(f"Test  R² (log): {test_r2_log:.4f}")
print(f"Gap:            {gap:.4f}")
if gap < 0.02:
    print("-> No significant overfitting.")
elif gap < 0.05:
    print("-> Slight overfitting. Consider reducing max_depth or n_estimators.")
else:
    print("-> Overfitting detected. Reduce model complexity.")

## 10. Feature Importance

In [ ]:
importances = best_gb.feature_importances_
feat_df = pd.DataFrame({
    'Feature':    X_train.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Show top 20
top20 = feat_df.head(20)

plt.figure(figsize=(10, 7))
sns.barplot(data=top20, x='Importance', y='Feature', palette='Blues_r')
plt.title('Gradient Boosting — Top 20 Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(top20.head(10).to_string(index=False))

## 11. Diagnostic Plots

**Residuals vs Predicted (log scale):** should be a flat random cloud centred on zero.
A funnel shape indicates heteroscedasticity; a curve indicates remaining non-linearity.

**Actual vs Predicted (original scale):** points should cluster tightly along the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted (log scale)
residuals = y_test.values - y_pred_test_log
axes[0].scatter(y_pred_test_log, residuals, alpha=0.3, s=10, color='steelblue')
axes[0].axhline(0, color='crimson', linewidth=1.5)
axes[0].set_xlabel('Predicted (log scale)')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted (log scale)')

# Actual vs Predicted (original scale)
axes[1].scatter(y_test_orig, y_pred_test_orig, alpha=0.3, s=10, color='darkorange')
lims = [min(y_test_orig.min(), y_pred_test_orig.min()),
        max(y_test_orig.max(), y_pred_test_orig.max())]
axes[1].plot(lims, lims, color='crimson', linewidth=1.5)
axes[1].set_xlabel('Actual Weekly Sales ($)')
axes[1].set_ylabel('Predicted Weekly Sales ($)')
axes[1].set_title(f'Actual vs Predicted  (R² = {test_r2:.3f})')

plt.suptitle('Gradient Boosting — Diagnostic Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Learning Curve

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_gb, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5, scoring='r2', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Training R²')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='steelblue')
plt.plot(train_sizes, val_mean, 'o-', color='darkorange', label='Validation R²')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='darkorange')
plt.xlabel('Training Set Size')
plt.ylabel('R²')
plt.title('Learning Curve — Gradient Boosting', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Final training R²:   {train_mean[-1]:.4f} ± {train_std[-1]:.4f}")
print(f"Final validation R²: {val_mean[-1]:.4f} ± {val_std[-1]:.4f}")

## 13. Comparison Against Linear Regression Baseline

In [ ]:
# Retrain Linear Regression on same split for fair comparison
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr_log  = lr.predict(X_test_sc)
y_pred_lr_orig = np.exp(y_pred_lr_log)

lr_r2_log   = r2_score(y_test, y_pred_lr_log)
lr_rmse_log = mean_squared_error(y_test, y_pred_lr_log) ** 0.5
lr_r2       = r2_score(y_test_orig, y_pred_lr_orig)
lr_mae      = mean_absolute_error(y_test_orig, y_pred_lr_orig)
lr_rmse     = mean_squared_error(y_test_orig, y_pred_lr_orig) ** 0.5

comparison = pd.DataFrame({
    'Model'       : ['Linear Regression (baseline)', 'Gradient Boosting'],
    'Test R² (log)' : [lr_r2_log, test_r2_log],
    'RMSE (log)'  : [lr_rmse_log, test_rmse_log],
    'Test R²'     : [lr_r2, test_r2],
    'Test MAE ($)': [f'${lr_mae:,.0f}', f'${test_mae:,.0f}'],
    'Test RMSE ($)': [f'${lr_rmse:,.0f}', f'${test_rmse:,.0f}']
})

print(comparison.to_string(index=False))

In [ ]:
# Bar chart comparison
metrics  = ['Test R² (log)', 'RMSE (log)']
lr_vals  = [lr_r2_log,   lr_rmse_log]
gb_vals  = [test_r2_log, test_rmse_log]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = ['Linear Regression', 'Gradient Boosting']
colors = ['#4878CF', '#E8713A']

for ax, metric, vals in zip(axes, metrics, [[lr_vals[i], gb_vals[i]] for i in range(2)]):
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Linear Regression vs Gradient Boosting', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()